# Import Packages

In [132]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import os 
from glob import glob

# work directory

In [133]:
wrk_directory ="C:/Users/l_v_v/Documents/GitHub/time_series_curuai/datasets/Parameters Time series/TSS Modeling"

# import Data

In [ ]:
paths = sorted(glob(os.path.join(wrk_directory,"*cv*.csv")))

cv_dfs = []

for cv_metric in paths:
    df_cv = pd.read_csv(cv_metric)
    df_cv['rmse'] = np.sqrt(df_cv['mse'])
    if 'Unnamed: 0' in df_cv.columns:
        df_cv = df_cv.drop(columns=['Unnamed: 0'])
    
    cv_dfs.append(df_cv)

final_cv_df = pd.concat(cv_dfs, ignore_index=True)


In [ ]:
# Identify unique model runs (excluding floating point metric columns to avoid duplication)
subset_cols = ['Model', 'Group', 'Feature']
if 'Params' in final_cv_df.columns:
    subset_cols.append('Params')
if 'water_period' in final_cv_df.columns:
    subset_cols.append('water_period')

# Drop duplicates
final_cv_df = final_cv_df.drop_duplicates(subset=subset_cols, keep='first')

cv_df = final_cv_df.loc[final_cv_df['water_period'].isna()].copy()

In [ ]:
period_cv = final_cv_df.dropna(subset='water_period')
period_cv

,Model,Group,Feature,Params,r2,mae,mse,mape,exp_var,rmse,water_period
69,ols,single_band,nir,NaN,0.750459,13.136463,4.484970e+02,0.286021,0.403573,21.177748,R
70,ols,single_band,red,NaN,0.468110,11.932755,4.286936e+02,0.256130,0.315162,20.704917,R
71,ols,single_band,nir_red_ratio,NaN,0.908369,13.853791,5.581768e+02,0.347170,0.400018,23.625766,R
72,ols,single_band,red_nir_ratio,NaN,0.542325,13.245558,4.979487e+02,0.309173,0.088449,22.314763,R
73,ols,multi_band,nir_red,NaN,0.688434,12.612278,4.277580e+02,0.267919,0.464074,20.682311,R
...,...,...,...,...,...,...,...,...,...,...,...
304,polynomial3,multi_band,nir_red_green,NaN,0.317686,66.027041,7.853916e+03,0.536376,0.343744,88.622324,LW
305,polynomial3,multi_band,all_bands,NaN,99.808216,285.634545,8.580621e+05,4.007965,88.781248,926.316436,LW
306,polynomial3,multi_band,location,NaN,548.006221,885.275299,3.527450e+06,9.384929,493.743708,1878.150577,LW
307,polynomial3,multi_band,period,NaN,99.808216,285.634545,8.580621e+05,4.007965,88.781248,926.316436,LW


# Sorting Model Metrics

In [155]:
def sort_models_weight(df, top_n=25):
    """
    Rank models based on best metric performance across multiple criteria.
    
    Uses a weighted composite score with cascading tiebreakers for unique rankings:
    - r2: weight 0.4 (higher is better)
    - Inverse MAE: weight 0.3 (lower error is better)
    - Inverse RMSE: weight 0.2 (lower error is better)
    - MAPE: weight 0.1 (lower is better)
    
    Tiebreakers (in order): r2, inverse MAE, inverse RMSE, inverse MAPE
    
    Filtering criteria:
    - r2: 0.4 < r2 <= 1.5 (valid range)
    - mae: <= 40
    - mape: <= 0.6
    - exp_var: >= 0.4
    
    Args:
        df: DataFrame with model metrics
        top_n: Return top N models (default 25)
    
    Returns:
        DataFrame with models ranked by composite score, showing score breakdown
    """
    df = df.copy()
    
    # Apply all filters at once to avoid row duplication
    initial_count = len(df)
    
    # Build filter mask
    mask = pd.Series([True] * len(df), index=df.index)
    
    if 'r2' in df.columns:
        mask &= (df['r2'] > 0.55) & (df['r2'] <= 1)
    
    if 'mae' in df.columns:
        mask &= (df['mae'] <= 23)
    
    if 'mape' in df.columns:
        mask &= (df['mape'] <= 0.6)
    
    if 'exp_var' in df.columns:
        mask &= (df['exp_var'] >= 0.4)
    
    df = df[mask].reset_index(drop=True)
    dropped_count = initial_count - len(df)
    
    if dropped_count > 0:
        print(f"Dropped {dropped_count} rows based on quality thresholds")
    
    if len(df) == 0:
        print("Warning: No rows remain after filtering!")
        return df
    
    # Calculate weighted composite score for unique ranking
    df['composite_score'] = 0.0
    
    # R2 component: weight 0.4 (higher is better, normalize to 0-1)
    if 'r2' in df.columns:
        r2_norm = (df['r2'] - df['r2'].min()) / (df['r2'].max() - df['r2'].min() + 1e-8)
        df['composite_score'] += r2_norm * 0.4
    
    # Inverse MAE component: weight 0.3 (lower error is better)
    if 'mae' in df.columns:
        mae_inv = 1.0 / (df['mae'] + 1e-8)
        mae_norm = (mae_inv - mae_inv.min()) / (mae_inv.max() - mae_inv.min() + 1e-8)
        df['composite_score'] += mae_norm * 0.3
    
    # Inverse RMSE component: weight 0.2 (lower error is better)
    if 'rmse' in df.columns:
        rmse_inv = 1.0 / (df['rmse'] + 1e-8)
        rmse_norm = (rmse_inv - rmse_inv.min()) / (rmse_inv.max() - rmse_inv.min() + 1e-8)
        df['composite_score'] += rmse_norm * 0.2
    
    # MAPE component: weight 0.1 (lower is better)
    if 'mape' in df.columns:
        mape_inv = 1.0 / (df['mape'] + 1e-8)
        mape_norm = (mape_inv - mape_inv.min()) / (mape_inv.max() - mape_inv.min() + 1e-8)
        df['composite_score'] += mape_norm * 0.1
    
    # Prepare tiebreaker columns (all ascending=False means descending)
    sort_cols = ['composite_score']
    ascending = [False]
    
    # Add cascading tiebreakers for truly unique ranking
    if 'r2' in df.columns:
        sort_cols.append('r2')
        ascending.append(False)
    
    if 'mae' in df.columns:
        sort_cols.append('mae')
        ascending.append(True)
    
    if 'mape' in df.columns:
        sort_cols.append('mape')
        ascending.append(True)
    
    # Add row index as final tiebreaker to guarantee uniqueness
    df['_row_index'] = range(len(df))
    sort_cols.append('_row_index')
    ascending.append(True)
    
    # Sort by composite score and tiebreakers
    df_sorted = df.sort_values(
        by=sort_cols,
        ascending=ascending
    ).reset_index(drop=True)
    
    # Drop helper columns before returning
    df_sorted = df_sorted.drop(columns=['_row_index'])
    
    # Return top N models (guaranteed unique due to cascading sort)
    return df_sorted.head(top_n)

# Example usage:
best_models = sort_models_weight(cv_df.loc[~cv_df['Feature'].isin(['location','period','period_location'])], top_n=5)
best_models[['Model', 'Feature', 'composite_score', 'mae', 'r2', 'mape','rmse']]

Dropped 19 rows based on quality thresholds


,Model,Feature,composite_score,mae,r2,mape,rmse
0,KRR,nir_red,0.777822,20.376511,0.697487,0.424498,33.645346
1,polynomial2,nir_red,0.741920,20.426360,0.691890,0.421390,34.091191
2,RF,all_bands,0.741103,20.121084,0.688698,0.432096,34.850323
3,SVM,all_bands,0.740704,19.890966,0.668301,0.357133,36.113694
4,SVM,nir_red,0.576256,20.779690,0.659198,0.375501,36.074798


In [156]:
def sort_models_count(df, top_n=25):
    """
    Rank models based on a 'Win Count' system.
    
    Score = Number of metrics for which this model holds the absolute best value 
            in the provided dataframe.
            - Higher is better for: r2, exp_var
            - Lower is better for: mae, mse, rmse, mape
            
    Sorting Order:
    1. Win Count (Descending)
    2. r2 (Descending) - Tiebreaker 1
    3. mae (Ascending) - Tiebreaker 2
    
    Args:
        df: DataFrame with model metrics
        top_n: Return top N models (default 25)
    
    Returns:
        DataFrame with models ranked by win count.
    """
    df = df.copy()
    
    # 1. Apply Quality Filters
    initial_count = len(df)
    mask = pd.Series([True] * len(df), index=df.index)
    
    if 'r2' in df.columns:
        mask &= (df['r2'] > 0.55) & (df['r2'] <= 1)
    if 'mae' in df.columns:
        mask &= (df['mae'] <= 23)
    if 'mape' in df.columns:
        mask &= (df['mape'] <= 0.6)
    if 'exp_var' in df.columns:
        mask &= (df['exp_var'] >= 0.4)
    
    df = df[mask].reset_index(drop=True)
    dropped_count = initial_count - len(df)
    
    if dropped_count > 0:
        print(f"Dropped {dropped_count} rows based on quality thresholds")
    
    if len(df) == 0:
        print("Warning: No rows remain after filtering!")
        return df
    
    # 2. Calculate "Win Count"
    df['metrics_won'] = 0
    
    # Define metrics and their optimization direction (True=Max is best, False=Min is best)
    metrics_config = {
        'r2': True,       # Higher is better
        'mae': False,     # Lower is better
        'rmse': False,    # Lower is better
        'mape': False     # Lower is better
    }
    
    for metric, higher_is_better in metrics_config.items():
        if metric in df.columns:
            if higher_is_better:
                best_val = df[metric].max()
                # Award point if value equals best (using close for float comparison safety)
                is_best = np.isclose(df[metric], best_val)
            else:
                best_val = df[metric].min()
                is_best = np.isclose(df[metric], best_val)
            
            # Increment score for winners
            df.loc[is_best, 'metrics_won'] += 1

    # 3. Sort
    # Primary: Win Count (Descending)
    # Secondary: R2 (Descending)
    # Tertiary: MAE (Ascending)
    sort_cols = ['metrics_won']
    ascending = [False]
    
    if 'r2' in df.columns:
        sort_cols.append('r2')
        ascending.append(False)
        
    if 'mae' in df.columns:
        sort_cols.append('mae')
        ascending.append(True)
        
    # Final tiebreaker: row index
    df['_row_index'] = range(len(df))
    sort_cols.append('_row_index')
    ascending.append(True)
    
    df_sorted = df.sort_values(
        by=sort_cols,
        ascending=ascending
    ).reset_index(drop=True)
    
    df_sorted = df_sorted.drop(columns=['_row_index'])
    
    return df_sorted.head(top_n)

# Example usage:
# Filters out location/period specific features to focus on spectral bands
best_models = sort_models_count(cv_df.loc[~cv_df['Feature'].isin(['location','period','period_location'])], top_n=5)

print("Top Models by Metric Wins:")
best_models[['Model', 'Feature', 'metrics_won', 'r2', 'mae', 'mape', 'rmse']]

Dropped 19 rows based on quality thresholds
Top Models by Metric Wins:


,Model,Feature,metrics_won,r2,mae,mape,rmse
0,SVM,all_bands,2,0.668301,19.890966,0.357133,36.113694
1,RF_GEE,all_bands,1,0.729764,22.266549,0.476356,39.967027
2,KRR,nir_red,1,0.697487,20.376511,0.424498,33.645346
3,polynomial2,nir_red,0,0.691890,20.426360,0.421390,34.091191
4,RF,all_bands,0,0.688698,20.121084,0.432096,34.850323
